In [1]:
# ==========================================
# Cell 1: 导入工具包与设置伪装参数
# ==========================================

# requests 是用来模拟浏览器发送网络请求的库 (拿取网页源码)
import requests 
# BeautifulSoup 是用来像切蛋糕一样解析复杂 HTML 网页的库 (提取我们需要的数据)
from bs4 import BeautifulSoup 

# 1. 伪装自己 (User-Agent)
# 建议把这里的字符串换成你在 chrome://version/ 里查到的你自己的 UA
# 这样服务器会认为：“哦，这是一个用 Windows 的真实玩家在用 Chrome 浏览器访问我。”
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0',
    'Accept-Language': 'en-US,en;q=0.9', # 告诉服务器我接受英文网页
    # 下面这行很重要，很多现代网站要求加上这个防盗链或来源标识
    'Referer': 'https://www.google.com/' 
}

# 目标 URL (CS2 职业选手参数列表页面)
TARGET_URL = 'https://prosettings.net/lists/cs2/'

print("✅ 设置完毕，准备就绪。")

✅ 设置完毕，准备就绪。


In [2]:
# ==========================================
# Cell 2: 发送请求，侦察敌情
# 技巧：如果这个 Cell 成功了，以后就不需要重复运行它了，直接解析 html_content 即可
# ==========================================

print(f"🕵️‍♂️ 开始对目标发送请求: {TARGET_URL}")

# requests.get 就像是你在浏览器输入网址并按下回车
# timeout=15 意味着如果 15 秒网站还没响应，就报错，防止程序死等
response = requests.get(TARGET_URL, headers=HEADERS, timeout=15)

# HTTP 状态码常识：200=成功，403=被拒绝，404=找不到页面，500=服务器崩溃
if response.status_code == 200:
    print("✅ 突破防线！成功获取网页响应。")
    
    # 将获取到的纯文本 HTML 代码保存到变量中
    html_content = response.text 
    
    # 召唤 BeautifulSoup 解析器，将文本转化为树状结构，方便我们后续按标签查找
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # 试着提取网页的标题 <title> 标签，验证是不是正确的页面
    title = soup.title.string if soup.title else "未找到标题"
    print(f"📌 获取到的网页标题为: {title}")
    
    # 看看我们到底抓下来多少个字符的代码（了解数据规模）
    print(f"📄 网页 HTML 源码总字符数: {len(html_content)}")

elif response.status_code == 403:
    print("❌ 遭到 403 拦截！网站启用了高级反爬 (可能是 Cloudflare/Datadome)。")
    print("返回内容片段:", response.text[:200]) # 打印一点错误信息看看原因

else:
    print(f"⚠️ 出现意外状态码: {response.status_code}")

🕵️‍♂️ 开始对目标发送请求: https://prosettings.net/lists/cs2/
✅ 突破防线！成功获取网页响应。
📌 获取到的网页标题为: CS2 Pro Settings and Gear List
📄 网页 HTML 源码总字符数: 1858999


In [3]:
# ==========================================
# Cell 3: 精确打击 - 分析选手数据的 HTML 结构
# ==========================================
import re

# 我们用当前 Top 级别的狙击手 ZywOo 作为“诱饵”来定位
target_player = "ZywOo" 
print(f"🔍 正在 1.8MB 的源码中搜索: {target_player} ...")

# 在 BeautifulSoup 解析后的树状结构中，寻找包含该名字的所有文本节点 (忽略大小写)
player_nodes = soup.find_all(string=re.compile(target_player, re.IGNORECASE))

if not player_nodes:
    print("❌ 没找到该选手！数据可能被隐藏在复杂的 JSON 中。")
else:
    print(f"🎯 找到了 {len(player_nodes)} 处包含 '{target_player}' 的文本！")
    print("-" * 50)
    
    # 提取第一处匹配的位置
    first_match = player_nodes[0]
    
    # 向上寻找它的“祖父”节点，通常这会包含这一整行（一整个选手）的数据
    # 使用 try-except 防止因为层级不够而报错
    try:
        # .parent 是一层一层向外找包裹它的 HTML 标签
        row_html = first_match.parent.parent.parent 
        
        print("💡 提取到该选手所在区块的 HTML 代码片段：\n")
        # .prettify() 可以把凌乱的代码排版得整整齐齐，[:800] 是为了只看前 800 个字符
        print(row_html.prettify()[:800]) 
        
    except Exception as e:
        print("提取 HTML 结构时出错:", e)

🔍 正在 1.8MB 的源码中搜索: ZywOo ...
🎯 找到了 10 处包含 'ZywOo' 的文本！
--------------------------------------------------
💡 提取到该选手所在区块的 HTML 代码片段：

<ol class="menu menu--players">
 <li class="menu-item">
  <a href="https://prosettings.net/players/donk/">
   <picture class="edge-images-container" style="--content-visibility: auto; --max-width: 30px; --width: 30px">
    <img alt="donk" class="attachment-30x30 size-30x30 is-prosettings-cf-local" decoding="async" height="30" src="https://prosettings.net/wp-content/uploads/donk-30x30-fitcontain-s1.webp" srcset="https://prosettings.net/wp-content/uploads/donk-30x30-fitcontain-s1.webp 30w, https://prosettings.net/wp-content/uploads/donk-30x30-15x-fitcontain-s1.webp 45w, https://prosettings.net/wp-content/uploads/donk-30x30-2x-fitcontain-s1.webp 60w" width="30"/>
   </picture>
   donk
  </a>
 </li>
 <li class="menu-item">
  <a href="https://prosettings.net/players/m0nesy/">
   <picture class=


In [4]:
# ==========================================
# Cell 4: 终极收割 - 提取表格数据并导出 CSV
# ==========================================
import pandas as pd

print("🚜 开始启动表格收割机...")

# 1. 找到网页中所有的 <tr> (表格行)
rows = soup.find_all('tr')

player_data_list = []
headers =[]

# 2. 遍历每一行，把数据提出来
for row in rows:
    # 提取当前行所有的单元格 <th> (表头) 或 <td> (具体数据)
    cells = row.find_all(['th', 'td'])
    
    # 提取纯文本内容，并去除首尾的换行符和空格
    # separator=' ' 是为了防止网页里隐藏的图标和文字粘连在一起
    cell_texts =[cell.get_text(separator=' ', strip=True) for cell in cells]
    
    # 如果这一行是空的，跳过
    if not cell_texts:
        continue
        
    # 3. 寻找表头 (Column Names)
    # 如果这一行里同时包含了 'Player' 和 'DPI'，那它肯定是表头！
    if 'Player' in cell_texts and 'DPI' in cell_texts:
        headers = cell_texts
        print(f"🎯 成功锁定表头: {headers}")
        
    # 4. 提取选手数据
    # 排除掉表头行，并且只要列数足够多的行（过滤掉网页上可能存在的其他乱七八糟的短表格）
    elif len(cell_texts) >= 5: 
        player_data_list.append(cell_texts)

print(f"✅ 成功提取了 {len(player_data_list)} 名职业选手的原始数据！")

# ---------------------------------------------------------
# 将散乱的列表转化为 Pandas DataFrame (数据分析的绝对核心)
# ---------------------------------------------------------

# 防止某些行的数据比表头短或长，我们做一个对齐处理
cleaned_data =[]
for data in player_data_list:
    if len(data) >= len(headers):
        cleaned_data.append(data[:len(headers)]) # 截断多余的列
    else:
        cleaned_data.append(data + [None] * (len(headers) - len(data))) # 不足的列用 None 补齐

# 创建 DataFrame！
df = pd.DataFrame(cleaned_data, columns=headers)

# 5. 导出为 CSV 文件
csv_filename = "cs2_pro_raw.csv"
# index=False 代表不要把行号 (0, 1, 2...) 写进文件
# encoding='utf-8-sig' 是为了防止带有特殊字符的名字（如一些俄语/北欧字母）在 Excel 里乱码
df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

print(f"💾 任务完成！数据已成功保存到本地：{csv_filename}")

# 6. 打印前 5 行数据预览一下战果
display(df.head())

🚜 开始启动表格收割机...
🎯 成功锁定表头: ['', 'Team', 'Player', 'Role', 'Mouse', 'HZ', 'DPI', 'Sens', 'eDPI', 'Zoom Sens', 'Monitor', 'GPU', 'Resolution', 'Aspect Ratio', 'Scaling Mode', 'Mousepad', 'Keyboard', 'Headset', 'Chair']
✅ 成功提取了 873 名职业选手的原始数据！
💾 任务完成！数据已成功保存到本地：cs2_pro_raw.csv


,,Team,Player,Role,Mouse,HZ,DPI,Sens,eDPI,Zoom Sens,Monitor,GPU,Resolution,Aspect Ratio,Scaling Mode,Mousepad,Keyboard,Headset,Chair
0,,Team Vitality,ZywOo,Sniper,Pulsar ZywOo The Chosen Mouse White,1000,400,2,800.00,1,ZOWIE XL2566K,RTX 3080,1280x960,4:3,Stretched,The Chosen Mousepad,ASUS ROG Falchion Ace HFX ZywOo Edition,ASUS ROG Pelta,Secretlab Titan Evo Vitality Edition
1,,Team Vitality,ropz,Rifler,Razer DeathAdder V3 HyperSpeed,2000,400,1.77,708,1,ZOWIE XL2586X+,RTX 4090,1920x1080,16:9,Native,Zowie Mousepad (Unreleased),ASUS ROG Falchion Ace 75 HE White,SteelSeries Arctis Nova Pro Wireless,Anthros V2
2,,Team Vitality,flameZ,Rifler,ZOWIE EC2-DW Grey (Unreleased),1000,400,3,1200.00,1.1,ZOWIE XL2566K,RTX 3080,1280x960,4:3,Stretched,Xtrfy GP4,ASUS ROG Falchion Ace 75 HE Black,Logitech G Pro X Headset,Secretlab Titan Evo Vitality Edition
3,,Team Vitality,apEX,Rifler,ZOWIE EC2-DW Black,1000,400,1.91,764.00,1,ZOWIE XL2566K,RTX 2080 Ti,1280x960,4:3,Stretched,ZOWIE G-TR,ASUS ROG Falchion Ace HFX Black,ASUS ROG Delta II,Secretlab Titan Evo Vitality Edition
4,,Team Vitality,mezii,Rifler,VAXEE XE V2 Fluorescent Green,2000,400,2.2,880.00,1,ZOWIE XL2566K,,1280x960,4:3,Stretched,BanKs Collection Heavy Claw by ESPTIGER,ASUS ROG Falchion Ace 75 HE White,ASUS ROG Pelta,Secretlab Titan Evo Vitality Edition


In [5]:
# ==========================================
# Cell 5: 获取所有选手的详情页 URL
# ==========================================
print("🔗 开始提取选手专属链接...")

# 创建一个字典，用来存 { "donk": "https://prosettings.net/players/donk/", ... }
player_urls = {}

# 重新遍历一次刚刚存好的 rows (包含所有 <tr> 标签)
for row in rows:
    # 找到这一行里所有的 <a> 标签 (也就是超链接)
    links = row.find_all('a')
    
    for link in links:
        href = link.get('href')
        # 确保这个链接存在，并且是通向选手个人页面的
        if href and 'players/' in href:
            player_name = link.get_text(strip=True)
            # 存入字典
            if player_name:
                player_urls[player_name] = href

print(f"✅ 成功提取了 {len(player_urls)} 个选手的个人主页链接！")

# 随便打印前 3 个看看对不对
# list(player_urls.items())[:3] 会把字典转成列表并切片前三个
print("🔗 链接示例预览：")
for name, url in list(player_urls.items())[:3]:
    print(f"{name} -> {url}")

🔗 开始提取选手专属链接...
✅ 成功提取了 872 个选手的个人主页链接！
🔗 链接示例预览：
ZywOo -> https://prosettings.net/players/zywoo/
ropz -> https://prosettings.net/players/ropz/
flameZ -> https://prosettings.net/players/flamez/


In [6]:
# ==========================================
# Cell 6: 探路详情页 (以 donk 为例)
# ==========================================
import time # 导入时间库，准备做防封禁停顿

# 我们拿 donk 开刀，如果你喜欢别的选手可以换他的名字
test_player = "donk"
# 如果前面成功拿到了链接，我们就用拿到的；如果没有，硬编码一个备用
test_url = player_urls.get(test_player, "https://prosettings.net/players/donk/")

print(f"🕵️‍♂️ 正在潜入 {test_player} 的个人主页: {test_url}")

# 发送请求
detail_response = requests.get(test_url, headers=HEADERS, timeout=15)

if detail_response.status_code == 200:
    print(f"✅ 成功进入 {test_player} 的详情页！")
    detail_soup = BeautifulSoup(detail_response.text, 'html.parser')
    
    # 【核心任务】：在个人主页里，所有的设置通常会被归类在一个个小盒子里
    # 比如 "Mouse Settings", "Video Settings", "Crosshair"
    # 我们试着找找网页里有没有包含 "Resolution" (分辨率) 的文本
    import re
    res_nodes = detail_soup.find_all(string=re.compile("Resolution", re.IGNORECASE))
    
    if res_nodes:
        print("\n🎯 找到有关 Resolution 的线索了！")
        first_res = res_nodes[0]
        # 向上找它的父节点，看看网页是怎么排版的
        try:
            res_html = first_res.parent.parent
            print("💡 包含分辨率区块的 HTML 代码片段：\n")
            print(res_html.prettify()[:500])
        except Exception as e:
            print("提取节点出错:", e)
    else:
        print("❌ 没找到 Resolution，页面结构可能比较特殊。")

else:
    print(f"❌ 请求失败，状态码: {detail_response.status_code}")

🕵️‍♂️ 正在潜入 donk 的个人主页: https://prosettings.net/players/donk/
✅ 成功进入 donk 的详情页！

🎯 找到有关 Resolution 的线索了！
💡 包含分辨率区块的 HTML 代码片段：

<tr class="format-select field-resolution" data-field="resolution">
 <th>
  Resolution
 </th>
 <td>
  1280x960
 </td>
</tr>



In [8]:
# ==========================================
# Cell 7 [最终完美版]: 解决覆盖问题并全量抓取！
# ==========================================
import time
import random
import pandas as pd

print("🚀 启动最终版全量收割机... (大概需要几分钟，请耐心等待)")

all_players_detailed_data = []

# 这次我们去掉 [:5]，直接遍历 player_urls 字典里的所有人！
total_targets = len(player_urls)

for index, (player_name, profile_url) in enumerate(player_urls.items()):
    # 打印进度条
    print(f"[{index+1}/{total_targets}] 抓取 {player_name} ...", end=" ")
    
    try:
        response = requests.get(profile_url, headers=HEADERS, timeout=10)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            player_info = {
                'Player': player_name,
                'Profile_URL': profile_url
            }
            
            detail_rows = soup.find_all('tr')
            for row in detail_rows:
                th = row.find('th')
                td = row.find('td')
                
                if th and td:
                    # 提取基础键值
                    original_key = th.get_text(strip=True)
                    value = td.get_text(strip=True)
                    
                    # 🌟 修复键冲突 Bug：如果字典里已经有这个键了，我们就加后缀
                    key = original_key
                    counter = 2
                    while key in player_info:
                        key = f"{original_key}_{counter}"
                        counter += 1
                        
                    # 安全存入字典
                    player_info[key] = value
            
            all_players_detailed_data.append(player_info)
            print("✅ OK")
            
        else:
            print(f"❌ 失败，状态码: {response.status_code}")
            
    except Exception as e:
        print(f"⚠️ 报错: {e}")
    
    # 防封禁机制，稍微停顿
    time.sleep(random.uniform(1.0, 2.5))

print("\n🎉 全部抓取完成！开始保存数据...")

df_detailed = pd.DataFrame(all_players_detailed_data)

# 保存为终极 RAW 数据
final_csv_name = "cs2_pro_detailed_RAW.csv"
df_detailed.to_csv(final_csv_name, index=False, encoding='utf-8-sig')

print(f"💾 完美！全部数据已保存至：{final_csv_name} (共 {len(df_detailed)} 行，{len(df_detailed.columns)} 列)")

# 预览一下修复后的前 5 行数据
display(df_detailed.head())

🚀 启动最终版全量收割机... (大概需要几分钟，请耐心等待)
[1/872] 抓取 ZywOo ... ✅ OK
[2/872] 抓取 ropz ... ✅ OK
[3/872] 抓取 flameZ ... ✅ OK
[4/872] 抓取 apEX ... ✅ OK
[5/872] 抓取 mezii ... ✅ OK
[6/872] 抓取 Jamppi ... ✅ OK
[7/872] 抓取 donk ... ✅ OK
[8/872] 抓取 sh1ro ... ✅ OK
[9/872] 抓取 magixx ... ✅ OK
[10/872] 抓取 zont1x ... ✅ OK
[11/872] 抓取 chopper ... ✅ OK
[12/872] 抓取 tN1R ... ✅ OK
[13/872] 抓取 Jimpphat ... ✅ OK
[14/872] 抓取 Spinx ... ✅ OK
[15/872] 抓取 Brollan ... ✅ OK
[16/872] 抓取 xertioN ... ✅ OK
[17/872] 抓取 torzsi ... ✅ OK
[18/872] 抓取 m0NESY ... ✅ OK
[19/872] 抓取 NiKo ... ✅ OK
[20/872] 抓取 TeSeS ... ✅ OK
[21/872] 抓取 kyxsan ... ✅ OK
[22/872] 抓取 kyousuke ... ✅ OK
[23/872] 抓取 bLitz ... ✅ OK
[24/872] 抓取 Techno4K ... ✅ OK
[25/872] 抓取 mzinho ... ✅ OK
[26/872] 抓取 910 ... ✅ OK
[27/872] 抓取 cobra ... ✅ OK
[28/872] 抓取 XANTARES ... ✅ OK
[29/872] 抓取 woxic ... ✅ OK
[30/872] 抓取 Wicadia ... ✅ OK
[31/872] 抓取 MAJ3R ... ✅ OK
[32/872] 抓取 Soulfly ... ✅ OK
[33/872] 抓取 b1t ... ✅ OK
[34/872] 抓取 w0nderful ... ✅ OK
[35/872] 抓取 iM ... ✅ OK
[36/872] 抓

,Player,Profile_URL,Name,Birthday,Country,Team,DPI,Sensitivity,eDPI,Zoom Sensitivity,...,Refraction Quality,Screenshot Quality,Ambient Occlusion_3,Local Reflections,Damage FX,Resolution_10,Aspect Ratio_6,Type_2,Thickness_3,Meshes
0,ZywOo,https://prosettings.net/players/zywoo/,Mathieu Herbaut,"November 9, 2000",France,Team Vitality,400,2,800.00,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ropz,https://prosettings.net/players/ropz/,Robin Kool,"December 22, 1999",Estonia,Team Vitality,400,1.77,708,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,flameZ,https://prosettings.net/players/flamez/,Shahar Shushan,"June 22, 2003",Israel,Team Vitality,400,3,1200.00,1.10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,apEX,https://prosettings.net/players/apex/,Dan Madesclaire,"February 22, 1993",France,Team Vitality,400,1.91,764.00,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mezii,https://prosettings.net/players/mezii/,William Merriman,"October 15, 1998",United Kingdom,Team Vitality,400,2.2,880.00,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
